[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# JSONB


## What you will be able to do

Put a Python dictionary into a JSONB column, which needs one wrapper because psycopg will not guess
that a dict means JSON. Say what `json` and `jsonb` are for and why nearly everything wants the
second. Ask whether a document contains another one, with `@>`, and reach into a document by path.
Read an `EXPLAIN` well enough to tell a sequential scan from an index scan, and see the same query
change from one to the other when a GIN index is added. And do the same through asyncpg, whose JSONB
needs the codec from **Types and Adaptation**.


## The idea

### The problem

`conn.execute("SELECT %s", ({"a": 1},))` does not work, and the reason is not an oversight. A Python
dictionary could be a JSON document, or an `hstore`, or a composite type, or a row. psycopg refuses
to choose, which is the same refusal **Types and Adaptation** met for a class of your own, arriving
for a type you would have thought was obvious.

So a dictionary is wrapped. One wrapper says JSON and the other says JSONB, and which one you pick
decides whether the column can be searched at all.

### What jsonb is

`json` stores the document as text, exactly as it was sent, whitespace and key order and duplicate
keys included. `jsonb` parses it into a binary form: keys sorted, duplicates dropped, whitespace
gone. That parsing is what lets PostgreSQL compare documents, index them, and reach into them
quickly.

The practical difference is that the operators worth having, `@>` most of all, exist for `jsonb` and
not for `json`. A column declared `json` looks identical until the first containment query, which
fails.

### Why it works that way

`@>` asks whether one document contains another, which is a question about structure rather than
about text. Answering it from stored text would mean parsing every row every time. Answering it from
the parsed form is a comparison, and the parsed form can be indexed, which is what GIN is for.

### Where this shows up

Anything with a shape that varies: event payloads, settings, an API response you want to keep whole.
The **JSON Columns** notebook of the **Peewee, Deep Dive** guide is the same subject through a
mapper, with SQLite's much smaller version of it underneath.

### What this notebook covers

The wrapper, and the two of them. `@>` and the other ways in. What `json` cannot do. `EXPLAIN`, read
well enough to see a plan change. A GIN index, and the same query before and after it. asyncpg's
side. Then the four failures.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import psycopg
from psycopg.types.json import Jsonb

with psycopg.connect("dbname=guide") as conn:
    try:
        conn.execute("SELECT %s::jsonb", ({"kind": "click"},))
    except psycopg.ProgrammingError as error:
        print("a bare dict:", error)
    conn.rollback()

    wrapped = conn.execute("SELECT %s", (Jsonb({"kind": "click"}),)).fetchone()
    print("wrapped:   ", wrapped, type(wrapped[0]).__name__)
    print("and it can be matched against:", conn.execute(
        "SELECT count(*) FROM events WHERE payload @> %s", (Jsonb({"size": 3}),)).fetchone()[0])
```

```
a bare dict: cannot adapt type 'dict' using placeholder '%s' (format: AUTO)
wrapped:    ({'kind': 'click'},) dict
and it can be matched against: 715
```

One wrapper on the way in, and nothing on the way back: the column loads as a dictionary without
being asked. The last line is the operator this notebook is really about, asking which of five
thousand documents contain `{"size": 3}`.


## Setup

Eleven imports, both drivers, the server, and a table with the same document stored twice.

- `psycopg` and `asyncpg` are the drivers, `errors` is the exception classes, and `Json` and `Jsonb`,
  from `psycopg.types.json`, are the two wrappers
- `json` is the standard library module, for the encoder and decoder asyncpg needs
- `subprocess`, `sys`, `os`, `getpass` and `time` stand the server up, which **A Server of Your Own**
  takes apart
- `version` and `PackageNotFoundError` install the drivers where they are missing

`docs` holds one row with a `jsonb` column and a `json` column containing the same document, which is
how the difference between them gets shown rather than described. `explain` prints a plan as node
names and an estimate, leaving out the costs and timings, which differ on every machine.


In [1]:
import getpass
import json
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors
from psycopg.types.json import Json, Jsonb

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def nodes(plan, depth=0):
    """The plan as a list of node names, without the costs and timings that differ per machine."""
    lines = ["  " * depth + plan["Node Type"]]
    for child in plan.get("Plans", []):
        lines += nodes(child, depth + 1)
    return lines


def explain(conn, query, params=None):
    """Print how the server would run a query, and how many rows it expects."""
    plan = conn.execute("EXPLAIN (FORMAT JSON) " + query, params).fetchone()[0][0]["Plan"]
    for line in nodes(plan):
        print("   ", line)
    print("    expecting about", plan["Plan Rows"], "rows")


def build_docs():
    """One table with the same document in a jsonb column and in a json column."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS docs")
        conn.execute("CREATE TABLE docs (id serial PRIMARY KEY, body jsonb, plain json)")
        conn.execute("INSERT INTO docs (body, plain) VALUES (%s, %s)",
                     (Jsonb({"kind": "click", "size": 3, "tags": ["a", "b"]}),
                      Json({"kind": "click", "size": 3, "tags": ["a", "b"]})))


print("server:", start_server())
print(report())
build_docs()
print("docs is ready")


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
docs is ready


## Worked examples

### The wrapper, and the two of them

`Jsonb` and `Json` differ in one thing, which is what the server calls the value:


In [2]:
document = {"kind": "click", "size": 3}

with psycopg.connect("dbname=guide") as conn:
    print("Jsonb ->", conn.execute("SELECT pg_typeof(%s)::text", (Jsonb(document),)).fetchone()[0])
    print("Json  ->", conn.execute("SELECT pg_typeof(%s)::text", (Json(document),)).fetchone()[0])
    print("both come back as:",
          type(conn.execute("SELECT %s", (Jsonb(document),)).fetchone()[0]).__name__)


Jsonb -> jsonb
Json  -> json
both come back as: dict


Reach for `Jsonb` unless you have a specific reason not to. `Json` is for a column that has to keep
the document byte for byte, which is rare and usually means the document is really a log line.

What the parsing costs and buys is visible in the stored text:


In [3]:
messy = {"b": 2, "a": 1, "a": 1}                                    # unordered, and a repeated key

with psycopg.connect("dbname=guide") as conn:
    print("as jsonb:", conn.execute("SELECT %s::text", (Jsonb(messy),)).fetchone()[0])
    print("as json: ", conn.execute("SELECT %s::text", (Json(messy),)).fetchone()[0])


as jsonb: {"a": 1, "b": 2}
as json:  {"b": 2, "a": 1}


The `jsonb` version has its keys in a different order and its whitespace gone, because it is not text
any more. That is the trade: you cannot get the original characters back, and in exchange the
document can be searched.

### Asking what a document contains

`@>` reads "contains". The right-hand side is a document, and the question is whether it is a subset
of the left:


In [4]:
with psycopg.connect("dbname=guide") as conn:
    for wanted in ({"size": 3}, {"n": 2}, {"size": 3, "n": 2}, {"size": 99}):
        found = conn.execute("SELECT count(*) FROM events WHERE payload @> %s",
                             (Jsonb(wanted),)).fetchone()[0]
        print(f"  payload @> {str(wanted):<34} {found:>5} rows")


  payload @> {'size': 3}                          715 rows
  payload @> {'n': 2}                               1 rows
  payload @> {'size': 3, 'n': 2}                    1 rows
  payload @> {'size': 99}                           0 rows


Containment is structural. `{"size": 3}` matches every document that has that key and value, whatever
else is in it, and adding a second key narrows it to the documents that have both. A value nothing
has matches nothing, without that being an error.

Note what is not in there: `kind` is a column of the table rather than a key of the document, so
`@> {"kind": "click"}` would match nothing at all. Containment asks about the document and knows
nothing about the row around it.

The other ways in are for when you want a value rather than a yes or no:


In [5]:
with psycopg.connect("dbname=guide") as conn:
    row = conn.execute("""
        SELECT payload -> 'size'        AS as_json,
               payload ->> 'size'       AS as_text,
               (payload ->> 'size')::int AS as_number,
               payload ? 'size'         AS has_size,
               jsonb_path_query_first(payload, '$.size') AS by_path
        FROM events ORDER BY id LIMIT 1""").fetchone()

for name, value in zip(("-> 'size'", "->> 'size'", "cast to int", "? 'size'", "path query"), row):
    print(f"  {name:<12} {type(value).__name__:<6} {value!r}")


  -> 'size'    int    2
  ->> 'size'   str    '2'
  cast to int  int    2
  ? 'size'     bool   True
  path query   int    2


`->` gives JSON and `->>` gives text, which is the same distinction the **JSON Columns** notebook of
the **Peewee, Deep Dive** guide runs into: comparing `-> 'size'` against a number compares JSON, and
the cast is what makes it arithmetic.

### What a json column cannot do

The same query, against the column declared `json`:


In [6]:
with psycopg.connect("dbname=guide") as conn:
    print("jsonb:", conn.execute("SELECT count(*) FROM docs WHERE body @> %s",
                                 (Jsonb({"kind": "click"}),)).fetchone()[0])
    try:
        conn.execute("SELECT count(*) FROM docs WHERE plain @> %s", (Jsonb({"kind": "click"}),))
    except errors.UndefinedFunction as error:
        print("json: ", type(error).__name__ + ":", str(error).splitlines()[0])


jsonb: 1
json:  UndefinedFunction: operator does not exist: json @> jsonb


The operator does not exist for that type. Nothing about the column looked wrong until this query,
which is what makes the choice at `CREATE TABLE` time worth getting right.

### Reading a plan

`EXPLAIN` says how the server would run a query. The whole output is long and machine specific, so
this notebook prints the node names and the estimate:


In [7]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("DROP INDEX IF EXISTS events_payload_idx")
    print("with no index:")
    explain(conn, "SELECT id FROM events WHERE payload @> %s", (Jsonb({"size": 3}),))


with no index:
    Seq Scan
    expecting about 707 rows


`Seq Scan` means every row is read and tested. For five thousand rows that is fine. The number after
it is the planner's estimate of how many rows will match, which comes from statistics and is a guess:


In [8]:
with psycopg.connect("dbname=guide") as conn:
    actual = conn.execute("SELECT count(*) FROM events WHERE payload @> %s",
                          (Jsonb({"size": 3}),)).fetchone()[0]
print("rows that actually match:", actual)


rows that actually match: 715


### The same query, with an index

GIN is the index type for containment. It indexes what is inside the document rather than the
document as a whole:


In [9]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("CREATE INDEX events_payload_idx ON events USING GIN (payload)")
    conn.execute("ANALYZE events")
    print("with a GIN index:")
    explain(conn, "SELECT id FROM events WHERE payload @> %s", (Jsonb({"size": 3}),))


with a GIN index:
    Bitmap Heap Scan
      Bitmap Index Scan
    expecting about 707 rows


`Seq Scan` has become `Bitmap Index Scan` feeding a `Bitmap Heap Scan`: the index is read to find
which rows might match, and only those rows are fetched. The estimate has not changed, because the
statistics have not: what changed is the work the server expects to do to find them.

An index is not free, and it is worth saying what it costs before reaching for one:


In [10]:
with psycopg.connect("dbname=guide") as conn:
    size = conn.execute("SELECT pg_size_pretty(pg_relation_size('events_payload_idx'))").fetchone()[0]
    table = conn.execute("SELECT pg_size_pretty(pg_relation_size('events'))").fetchone()[0]
print("the table:", table, "| the index:", size)
print("and every write to events now has to update it too")


the table: 496 kB | the index: 312 kB
and every write to events now has to update it too


There is a second GIN operator class, `jsonb_path_ops`, which indexes only containment and is smaller
and faster for it, at the cost of not supporting the key-existence operator `?`. That is the usual
choice when the queries are all `@>`.

### asyncpg, and the codec again

asyncpg has no JSONB rule of its own, which **Types and Adaptation** showed. In this notebook that
means one registration before any of the above works:


In [11]:
conn = await asyncpg.connect(database="guide")
print("before:", type(await conn.fetchval("SELECT payload FROM events LIMIT 1")).__name__)

await conn.set_type_codec("jsonb", encoder=json.dumps, decoder=json.loads, schema="pg_catalog")
print("after: ", type(await conn.fetchval("SELECT payload FROM events LIMIT 1")).__name__)

found = await conn.fetchval("SELECT count(*) FROM events WHERE payload @> $1", {"size": 3})
print("and containment works:", found)
await conn.close()


before: str
after:  dict
and containment works: 715


With the codec registered, a Python dictionary is what goes in and what comes out, and no wrapper is
needed: asyncpg was told what `jsonb` means, where psycopg is told per value.

### When to reach for which

| What you want | How to write it |
|---|---|
| a document in a column | `jsonb`, and `Jsonb(value)` as the parameter |
| the document kept byte for byte | `json`, and `Json(value)` |
| does this document contain that one | `payload @> %s` |
| a value out, as JSON | `payload -> 'key'` |
| a value out, as text | `payload ->> 'key'` |
| a number out | `(payload ->> 'key')::int` |
| does this key exist | `payload ? 'key'` |
| containment to use an index | `CREATE INDEX ... USING GIN (payload)` |
| a smaller index, containment only | `USING GIN (payload jsonb_path_ops)` |
| the same in asyncpg | `set_type_codec("jsonb", ...)`, then plain dictionaries |

`jsonb` and `Jsonb` are the defaults and `json` is the exception. A GIN index is worth adding when a
containment query is run often enough to matter, and not before, because it costs space and slows
every write to the table.

### A search over documents, finished

Everything above, as the thing it is for: a filter built from whatever keys the caller gave, matched
by containment, with the plan printed so the index is visibly doing its job.


In [12]:
def find(limit=3, **wanted):
    """Events whose payload contains all of these keys and values."""
    with psycopg.connect("dbname=guide") as conn:
        return conn.execute(
            "SELECT id, payload FROM events WHERE payload @> %s ORDER BY id LIMIT %s",
            (Jsonb(wanted), limit)).fetchall()


for row in find(size=3):
    print("  ", row)
print()
print("both keys at once:", len(find(size=3, n=2, limit=50)), "row")

with psycopg.connect("dbname=guide") as conn:
    print("and the plan for it:")
    explain(conn, "SELECT id FROM events WHERE payload @> %s", (Jsonb({"size": 3, "n": 2}),))


   (2, {'n': 2, 'size': 3})
   (9, {'n': 9, 'size': 3})
   (16, {'n': 16, 'size': 3})

both keys at once: 1 row
and the plan for it:
    Bitmap Heap Scan
      Bitmap Index Scan
    expecting about 1 rows


The filter is a dictionary built by the caller, wrapped once, and matched against every document with
one operator. Adding a key narrows the result without changing the query text, which is the thing a
column of documents is for.

### Where each part came from

| In the search | What it relies on | The section that showed it |
|---|---|---|
| `Jsonb(wanted)` | a dictionary psycopg will accept | The wrapper |
| `payload @> %s` | containment, structural rather than textual | Asking what a document contains |
| the plan showing a `Bitmap Index Scan` | the GIN index | The same query, with an index |
| `%s` for the limit too | a value is a value | **Placeholders and Identifiers** |
| the rows coming back as dictionaries | the loader that is already there | The wrapper |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/06-jsonb-solutions.ipynb).

**1.** Try to pass a dictionary as a parameter, then do it the way that works, and print what the
server calls each of the two wrappers.


In [13]:
# your code here


**2.** Count the events whose payload contains `{"kind": "purchase"}`, and then the ones that also
contain `{"size": 5}`.


In [14]:
# your code here


**3.** Print one event's `size` as JSON, as text and as a number, and show the type of each.


In [15]:
# your code here


**4.** Show that the containment operator does not exist for the `json` column in `docs`.


In [16]:
# your code here


**5.** Print the plan for a containment query with the GIN index dropped and again with it created.


In [17]:
# your code here


**6.** Run the same containment query through asyncpg, registering whatever it needs first.


In [18]:
# your code here


## Common errors

### psycopg.ProgrammingError: cannot adapt type 'dict' using placeholder '%s' (format: AUTO)


In [19]:
with psycopg.connect("dbname=guide") as conn:
    conn.execute("INSERT INTO docs (body) VALUES (%s)", ({"kind": "click"},))


ProgrammingError: cannot adapt type 'dict' using placeholder '%s' (format: AUTO)

The same message **Types and Adaptation** gave for a class of your own, for the most ordinary type in
Python. psycopg is not being difficult: a dictionary is the natural Python shape for four different
PostgreSQL types, and choosing silently would be a worse failure than this one.

The wrapper is the whole fix, and it can be made the default for a connection if writing it every
time is tiresome:


In [20]:
with psycopg.connect("dbname=guide") as conn:
    conn.execute("INSERT INTO docs (body) VALUES (%s)", (Jsonb({"kind": "click"}),))
    print("wrapped:", conn.execute("SELECT count(*) FROM docs").fetchone()[0], "rows")
    conn.rollback()

    conn.adapters.register_dumper(dict, psycopg.types.json.JsonbDumper)
    conn.execute("INSERT INTO docs (body) VALUES (%s)", ({"kind": "click"},))
    print("with dict registered as jsonb:", conn.execute("SELECT count(*) FROM docs").fetchone()[0],
          "rows")
    conn.rollback()


wrapped: 2 rows
with dict registered as jsonb: 2 rows


Registering `dict` like that is a decision for a program that only ever means JSON by a dictionary.
It is the right call surprisingly often, and it is worth writing down where somebody will find it.

### psycopg.errors.UndefinedFunction: operator does not exist: json @> jsonb


In [21]:
with psycopg.connect("dbname=guide") as conn:
    conn.execute("SELECT count(*) FROM docs WHERE plain @> %s", (Jsonb({"kind": "click"}),))


UndefinedFunction: operator does not exist: json @> jsonb
LINE 1: SELECT count(*) FROM docs WHERE plain @> $1
                                              ^
HINT:  No operator matches the given name and argument types. You might need to add explicit type casts.

The column is `json` and containment is a `jsonb` operator. The message is precise about it: there is
no `@>` taking a `json` on the left.

A cast gets you through one query, and changing the column is the real answer, because a cast on
every row is a sequential scan that no index can help:


In [22]:
with psycopg.connect("dbname=guide") as conn:
    print("cast per row:", conn.execute(
        "SELECT count(*) FROM docs WHERE plain::jsonb @> %s", (Jsonb({"kind": "click"}),)).fetchone()[0])
    print("the plan for that:")
    explain(conn, "SELECT id FROM docs WHERE plain::jsonb @> %s", (Jsonb({"kind": "click"}),))


cast per row: 1
the plan for that:
    Seq Scan
    expecting about 8 rows


### psycopg.errors.UndefinedFunction: operator does not exist: jsonb > integer


In [23]:
with psycopg.connect("dbname=guide") as conn:
    conn.execute("SELECT count(*) FROM events WHERE payload -> 'size' > 3")


UndefinedFunction: operator does not exist: jsonb > integer
LINE 1: SELECT count(*) FROM events WHERE payload -> 'size' > 3
                                                            ^
HINT:  No operator matches the given name and argument types. You might need to add explicit type casts.

`->` gives back JSON, and there is no operator for comparing JSON with an integer. The message says
exactly that, naming both sides.

`->>` gives text, which does have a `>`, and that is the trap: it compares as text. With this data
the two agree, because every size is a single digit, and single digits sort the same either way.
They stop agreeing the moment a value has two:


In [24]:
with psycopg.connect("dbname=guide") as conn:
    print("as text, '10' > '9' is:  ", conn.execute("SELECT '10' > '9'").fetchone()[0])
    print("as numbers, 10 > 9 is:  ", conn.execute("SELECT 10 > 9").fetchone()[0])
    print()
    print("text comparison here:   ", conn.execute(
        "SELECT count(*) FROM events WHERE payload ->> 'size' > '3'").fetchone()[0])
    print("with the cast:          ", conn.execute(
        "SELECT count(*) FROM events WHERE (payload ->> 'size')::int > 3").fetchone()[0])


as text, '10' > '9' is:   False
as numbers, 10 > 9 is:   True

text comparison here:    2856
with the cast:           2856


### No error, and a containment query that reads every row: a GIN index nobody made


In [25]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("DROP INDEX IF EXISTS events_payload_idx")
    print("no index:")
    explain(conn, "SELECT id FROM events WHERE payload @> %s", (Jsonb({"size": 3}),))

    conn.execute("CREATE INDEX events_payload_idx ON events USING GIN (payload)")
    conn.execute("ANALYZE events")
    print("with one:")
    explain(conn, "SELECT id FROM events WHERE payload @> %s", (Jsonb({"size": 3}),))


no index:
    Seq Scan
    expecting about 707 rows
with one:
    Bitmap Heap Scan
      Bitmap Index Scan
    expecting about 707 rows


Both queries return the same rows, so nothing looks wrong. The difference is `Seq Scan` against
`Bitmap Index Scan`, which on five thousand rows is nothing and on five million is the difference
between a page loading and a page timing out.

This is the one in this list that will not announce itself. The way to find it is to read the plan
for the queries that matter, which is why `EXPLAIN` is in this notebook at all.


In [26]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("DROP INDEX IF EXISTS events_payload_idx")         # leave the table as it was found
    print("index dropped, table left as Setup made it")


index dropped, table left as Setup made it


## Recap

- psycopg will not treat a dictionary as JSON without being told. `Jsonb(value)` is the wrapper, and
  `Json(value)` is the one for a column declared `json`.
- `jsonb` is parsed, so its keys are sorted, duplicates are gone and whitespace is lost. `json` keeps
  the text exactly and cannot be searched with the operators worth having.
- `@>` asks whether one document contains another, structurally. `->` gives JSON, `->>` gives text,
  and a number needs a cast on top of `->>`.
- `EXPLAIN (FORMAT JSON)` gives a plan you can read from code. `Seq Scan` reads every row;
  `Bitmap Index Scan` reads an index first.
- A GIN index makes containment an index scan, costs space, and slows writes.
  `jsonb_path_ops` is the smaller version for containment alone.
- A parameter with no type can leave an operator ambiguous, which is what `jsonb ? unknown` means,
  and a cast resolves it.
- asyncpg needs `set_type_codec` for JSONB, after which dictionaries work in both directions with no
  wrapper.


## What is next

The **Server-Side Cursors** notebook is about where the rows are: `execute` has already pulled the
whole result into your process before `fetchall` runs, so fetching in batches saves nothing, and a
named cursor is the thing that does.


---

&#8592; **Previous:** [Types and Adaptation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/05-types-and-adaptation.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
